# Module 1 — Dataset Collection & Validation (Google Colab mirror)

**Explainable Multilingual Fake News Detection using XLM-RoBERTa, Supervised Contrastive Learning, Fact Verification, and Explainable AI for Low-Resource Languages**

This notebook mirrors the local Module 1 pipeline exactly — it imports and runs the
same `src/data_collection/*` code and `scripts/*.py` entry points that run locally, so
there is only one implementation of the logic to keep correct (this notebook does not
re-implement collection/validation logic; it only orchestrates the existing scripts).

**What this notebook does, in order:**
1. Mounts Google Drive.
2. Locates the project folder inside Drive (so every output — raw data, processed
   data, logs, reports — is written straight to Drive and survives the Colab runtime
   being recycled).
3. Installs Module 1's Python dependencies.
4. Runs the English collectors (WELFake, ISOT).
5. Runs the Tamil collectors (BOOM Tamil, Daily Thanthi, + best-effort others).
6. Runs validation / deduplication / language identification / label standardization
   / statistics reporting.
7. Prints a final summary so you can confirm the run before doing anything else.

**Before running:** upload/sync this entire project folder (`Fake_News_Detection/`,
the same one on your local machine) into your Google Drive — e.g.
`My Drive/Fake_News_Detection`. The easiest way is the Google Drive desktop app's
folder sync, or a manual upload of the whole folder via drive.google.com. This
notebook does **not** run automatically on open — you must run each cell yourself
(Runtime → Run all, or cell-by-cell), per the project's requirement that nothing
executes without an explicit go-ahead.

**GPU note:** Module 1 is pure CPU work (HTTP requests, pandas, regex/langdetect) —
no GPU is required for this notebook. Set the Colab runtime to CPU (Runtime → Change
runtime type → CPU) to avoid using up GPU quota you'll want for Module 3 onward.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Locate the project folder

Edit `DRIVE_PROJECT_PATH` below if you placed the folder somewhere other than
`My Drive/Fake_News_Detection`.

In [ ]:
import os
import sys
from pathlib import Path

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Fake_News_Detection"

PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found at {PROJECT_ROOT}. "
        "Upload/sync the Fake_News_Detection folder to this path in Google Drive, "
        "or edit DRIVE_PROJECT_PATH above to match where you placed it."
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory:", os.getcwd())


## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 4. Step 2 — English dataset collectors (WELFake, ISOT)

Downloads both datasets directly (Zenodo / University of Victoria). If a dataset was
already downloaded in a previous Colab session (and Drive still has the file), the
collector detects this and skips re-downloading — safe to re-run this cell.

In [ ]:
!python scripts/collect_english.py --sources welfake isot


## 5. Step 3 — Tamil dataset collection

Runs every registered Tamil collector. Sources that are unverified placeholders or
currently uncollectable (see `docs/module1_overview.md`) fail individually and are
reported in the summary table without stopping the others. Adjust `--max-pages`
lower for a quick smoke test, or remove it (defaults apply) for a full run.

This step can take a while for `dailythanthi_sitemap` (crawls real sitemap pages with
polite rate-limiting) — reduce `--max-daily-sitemaps` / `--max-articles` for a
shorter run.

In [ ]:
!python scripts/collect_tamil.py --all --max-pages 20 --max-daily-sitemaps 10 --max-articles 500


## 6. Step 4 — Validation, deduplication, language identification, label standardization, statistics

Runs the same pipeline as `python scripts/validate_and_standardize.py` locally.
Reads everything under `datasets/raw/`, writes the standardized dataset to
`datasets/processed/`, and writes the validation + statistics reports to
`outputs/module1/` — all inside your Drive-mounted project folder, so these
persist after the Colab runtime disconnects.

In [ ]:
!python scripts/validate_and_standardize.py


## 7. Verify outputs

In [ ]:
import json

stats_path = PROJECT_ROOT / "outputs" / "module1" / "dataset_statistics.json"
report_path = PROJECT_ROOT / "outputs" / "module1" / "validation_report.json"

with open(stats_path, encoding="utf-8") as f:
    stats = json.load(f)
with open(report_path, encoding="utf-8") as f:
    report = json.load(f)

print("=== Dataset statistics ===")
print(json.dumps(stats, indent=2, ensure_ascii=False)[:2000])

print("\n=== Validation report stages ===")
print(json.dumps(report["stages"], indent=2, ensure_ascii=False, default=str))

processed_dir = PROJECT_ROOT / "datasets" / "processed"
print("\nFiles in datasets/processed/:")
for p in sorted(processed_dir.glob("*")):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")


## Done

Module 1 is complete once the cells above run without error and the statistics/
validation reports look sane. Everything written by this notebook lives under your
Drive-mounted project folder (`datasets/`, `outputs/module1/`, `logs/`), identical in
structure to a local run — Module 2 (and this notebook's future siblings for Modules
3+, which will need a GPU runtime) can pick up from `datasets/processed/` either
locally or in a fresh Colab session against the same Drive folder.

Do not proceed to Module 2 until you've reviewed these outputs and given explicit
approval, per this project's module-by-module workflow.